# Build Cross-Rotation Splits

Generate the official 8-fold cross-rotation split CSVs from the processed manifest.

In [8]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_1 = start_notebook_cell_progress('03_build_cross_rotation_splits.ipynb', 'Load shared setup', total_steps=1)

import json
import re
from pathlib import Path

import pandas as pd

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

finish_notebook_cell_progress(NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_1)


[START] 03_build_cross_rotation_splits.ipynb | Load shared setup [0/1 step] elapsed=0.0s
[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [0/1 step] elapsed=0.0s
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [1/1 step] elapsed=0.0s
[RUNNING] 03_build_cross_rotation_splits.ipynb | Load shared setup | done [0/1 step] elapsed=0.0s
[RUNNING] 03_build_cross_rotation_splits.ipynb | Load shared setup | done [1/1 step] elapsed=0.0s


In [9]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_2 = start_notebook_cell_progress('03_build_cross_rotation_splits.ipynb', 'Define split helpers', total_steps=1)

def sample_sort_key(sample_id: str) -> tuple[int, str]:
    match = re.search(r'(\d+)$', str(sample_id))
    if match:
        return int(match.group(1)), str(sample_id)
    return 10**9, str(sample_id)


def require_official_sample_ids(manifest_df: pd.DataFrame, expected_count: int = 8) -> pd.DataFrame:
    if 'sample_id' not in manifest_df.columns:
        raise ValueError('Processed manifest must contain a sample_id column.')
    df = manifest_df.copy()
    df['sample_id'] = df['sample_id'].astype(str).str.strip()
    if not df['sample_id'].all():
        raise ValueError('Every processed row must have a non-empty sample_id.')
    unique_sample_ids = sorted(df['sample_id'].unique(), key=sample_sort_key)
    if len(unique_sample_ids) != expected_count:
        raise ValueError(
            f'Official cross-rotation requires exactly {expected_count} unique sample_ids. Found {len(unique_sample_ids)}.'
        )
    return df


from sklearn.model_selection import StratifiedKFold


def build_roboflow_stratified_8fold_splits(
    processed_df: pd.DataFrame,
    output_root: Path,
) -> tuple[dict[str, Path], pd.DataFrame, pd.DataFrame]:
    required_columns = {'roboflow_split', 'image_file_name', 'label', 'local_image_path'}
    missing = required_columns - set(processed_df.columns)
    if missing:
        raise ValueError(f'Roboflow processed manifest is missing columns: {sorted(missing)}')
    df = processed_df.reset_index(drop=True).copy()
    df['roboflow_split'] = df['roboflow_split'].astype(str).str.strip().str.lower()
    if set(df['roboflow_split']) != {'train', 'valid', 'test'}:
        raise ValueError('Roboflow processed manifest must retain train, valid, and test provenance rows.')
    if not df['local_image_path'].astype(str).is_unique:
        raise ValueError('Roboflow processed manifest must have unique local_image_path values for 8-fold splitting.')
    label_counts = df['label'].astype(str).value_counts()
    if label_counts.empty or int(label_counts.min()) < 8:
        raise ValueError('Roboflow 8-fold stratification requires at least 8 rows per label.')
    ensure_dir(output_root)
    written_paths: dict[str, Path] = {}
    all_sampled_images_path = output_root / 'all_sampled_images.csv'
    df.to_csv(all_sampled_images_path, index=False)
    written_paths['all_sampled_images'] = all_sampled_images_path

    stratifier = StratifiedKFold(n_splits=8, shuffle=True, random_state=2026)
    partitions = [test_indices for _, test_indices in stratifier.split(df, df['label'])]
    summary_rows: list[dict[str, object]] = []
    leakage_rows: list[dict[str, object]] = []
    all_indices = set(range(len(df)))
    for fold_index in iter_notebook_progress(range(1, 9), '03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds', total=8, unit='fold', leave=True):
        test_indices = set(partitions[fold_index - 1])
        val_indices = set(partitions[fold_index % 8])
        train_indices = all_indices - test_indices - val_indices
        fold_name = f'fold{fold_index}'
        split_frames = {
            'train': df.iloc[sorted(train_indices)].copy(),
            'val': df.iloc[sorted(val_indices)].copy(),
            'test': df.iloc[sorted(test_indices)].copy(),
        }
        for split_name, split_df in split_frames.items():
            split_df['fold'] = fold_name
            split_df['split'] = split_name
            split_df['split_type'] = 'roboflow_stratified_8fold'
            output_path = output_root / f'{fold_name}_{split_name}.csv'
            split_df.to_csv(output_path, index=False)
            written_paths[f'{fold_name}_{split_name}'] = output_path
        summary_rows.append({
            'fold': fold_name,
            'train_samples': 'stratified_train',
            'val_sample': f'partition{(fold_index % 8) + 1}',
            'test_sample': f'partition{fold_index}',
            'train_count': len(split_frames['train']),
            'val_count': len(split_frames['val']),
            'test_count': len(split_frames['test']),
        })
        identity_sets = {name: set(frame['local_image_path'].astype(str)) for name, frame in split_frames.items()}
        leakage_rows.append({
            'fold': fold_name,
            'train_val_overlap': bool(identity_sets['train'] & identity_sets['val']),
            'train_test_overlap': bool(identity_sets['train'] & identity_sets['test']),
            'val_test_overlap': bool(identity_sets['val'] & identity_sets['test']),
        })

    summary_df = pd.DataFrame(summary_rows)
    leakage_df = pd.DataFrame(leakage_rows)
    summary_path = output_root / 'cross_rotation_summary.csv'
    leakage_path = output_root / 'cross_rotation_leakage_check.csv'
    summary_df.to_csv(summary_path, index=False)
    leakage_df.to_csv(leakage_path, index=False)
    written_paths['summary'] = summary_path
    written_paths['leakage'] = leakage_path
    return written_paths, summary_df, leakage_df


def build_official_cross_rotation_splits(
    processed_df: pd.DataFrame,
    output_root: Path,
) -> tuple[dict[str, Path], pd.DataFrame, pd.DataFrame]:
    df = require_official_sample_ids(processed_df)
    ensure_dir(output_root)

    unique_sample_ids = sorted(df['sample_id'].unique(), key=sample_sort_key)
    summary_rows: list[dict[str, object]] = []
    leakage_rows: list[dict[str, object]] = []
    written_paths: dict[str, Path] = {}

    all_sampled_images_path = output_root / 'all_sampled_images.csv'
    df.to_csv(all_sampled_images_path, index=False)
    written_paths['all_sampled_images'] = all_sampled_images_path

    for fold_index, test_sample_id in enumerate(
        iter_notebook_progress(
            unique_sample_ids,
            '03_build_cross_rotation_splits.ipynb | build official folds',
            total=len(unique_sample_ids),
            unit='fold',
            leave=True,
        ),
        start=1,
    ):
        val_sample_id = unique_sample_ids[fold_index % len(unique_sample_ids)]
        train_sample_ids = [sample_id for sample_id in unique_sample_ids if sample_id not in {test_sample_id, val_sample_id}]

        fold_name = f'fold{fold_index}'
        train_df = df[df['sample_id'].isin(train_sample_ids)].copy()
        val_df = df[df['sample_id'] == val_sample_id].copy()
        test_df = df[df['sample_id'] == test_sample_id].copy()

        for split_name, split_df in (('train', train_df), ('val', val_df), ('test', test_df)):
            split_df['fold'] = fold_name
            split_df['split'] = split_name
            split_df['split_type'] = 'cross_rotation'
            output_path = output_root / f'{fold_name}_{split_name}.csv'
            split_df.to_csv(output_path, index=False)
            written_paths[f'{fold_name}_{split_name}'] = output_path

        summary_rows.append(
            {
                'fold': fold_name,
                'train_samples': '|'.join(train_sample_ids),
                'val_sample': val_sample_id,
                'test_sample': test_sample_id,
                'train_count': len(train_df),
                'val_count': len(val_df),
                'test_count': len(test_df),
            }
        )

        leakage_rows.append(
            {
                'fold': fold_name,
                'train_val_overlap': bool(set(train_df['sample_id']) & set(val_df['sample_id'])),
                'train_test_overlap': bool(set(train_df['sample_id']) & set(test_df['sample_id'])),
                'val_test_overlap': bool(set(val_df['sample_id']) & set(test_df['sample_id'])),
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    leakage_df = pd.DataFrame(leakage_rows)
    summary_path = output_root / 'cross_rotation_summary.csv'
    leakage_path = output_root / 'cross_rotation_leakage_check.csv'
    summary_df.to_csv(summary_path, index=False)
    leakage_df.to_csv(leakage_path, index=False)
    written_paths['summary'] = summary_path
    written_paths['leakage'] = leakage_path
    return written_paths, summary_df, leakage_df

finish_notebook_cell_progress(NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_2)


[START] 03_build_cross_rotation_splits.ipynb | Define split helpers [0/1 step] elapsed=0.0s
[RUNNING] 03_build_cross_rotation_splits.ipynb | Define split helpers | done [0/1 step] elapsed=0.0s
[RUNNING] 03_build_cross_rotation_splits.ipynb | Define split helpers | done [1/1 step] elapsed=0.0s


In [10]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_3 = start_notebook_cell_progress('03_build_cross_rotation_splits.ipynb', 'Build official folds', total_steps=1)

PROCESSED_MANIFEST_PATH = Path(str(override('PROCESSED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'processed_manifest.csv')))
CROSS_ROTATION_OUTPUT_ROOT = Path(str(override('CROSS_ROTATION_OUTPUT_ROOT', GENERATED_SPLITS_ROOT)))

processed_df = pd.read_csv(PROCESSED_MANIFEST_PATH, dtype=str).fillna('')
split_builder = build_roboflow_stratified_8fold_splits if DATASET_SOURCE == 'roboflow' else build_official_cross_rotation_splits
written_paths, summary_df, leakage_df = split_builder(
    processed_df,
    output_root=CROSS_ROTATION_OUTPUT_ROOT,
)

print(f'Generated fold files: {len(written_paths)}')
print(summary_df[['fold', 'val_sample', 'test_sample']].to_string(index=False))

finish_notebook_cell_progress(NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_3)


[START] 03_build_cross_rotation_splits.ipynb | Build official folds [0/1 step] elapsed=0.0s
[START] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [0/8 fold] elapsed=0.0s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [1/8 fold] elapsed=0.2s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [2/8 fold] elapsed=0.4s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [3/8 fold] elapsed=0.6s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [4/8 fold] elapsed=0.8s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [5/8 fold] elapsed=1.0s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [6/8 fold] elapsed=1.1s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [7/8 fold] elapsed=1.3s
[RUNNING] 03_build_cross_rotation_splits.ipynb | build Roboflow 8-folds [8/8 fold] elapsed=1.5s
Generated fold files: 27
 fold val_sample test